In [1]:
import numpy as np
import pandas as pd
import polars as pl
import svy
from statsmodels.stats.multitest import multipletests

In [2]:
h5_markers = pd.read_csv('../data/processed/h5_markers.csv')

def enrich(sample_path, weight_divisor):
    df = pd.read_csv(sample_path)
    df = df[~df['ft3_implausible']]
    df = pd.merge(df, h5_markers, on=['SEQN', 'cycle'], how='left')
    df['wt_pooled'] = df['wt_thyroid_2yr'] / weight_divisor
    return df

In [3]:
# H1 population -> pulse vs triglycerides, plus the pulse-vs-free-T3 calibration check
h1 = enrich('../data/processed/h1_analytic_sample.csv', 3)
h1_nobb = h1[~h1['on_beta_blocker']]

design1 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wt_pooled')
sample1 = svy.Sample(pl.from_pandas(h1_nobb), design=design1)

model_pulse_tg = svy.GLM(sample1)
model_pulse_tg.fit(y='LBXTR', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print('=== pulse -> triglycerides ===')
print(model_pulse_tg)

model_calibration = svy.GLM(sample1)
model_calibration.fit(y='BPXPLS', x=['LBXT3F'], family='gaussian')
print()
print('=== calibration: pulse ~ free T3 (descriptive, not part of the formal multiplicity family) ===')
print(model_calibration)

=== pulse -> triglycerides ===
GLM: Gaussian (identity)
  Modeling : LBXTR
  n            : 2924          DF Residuals : 45
  Deviance     : 2.9246e+07    Scale        : 624828.3345
  AIC          : 35253.6660    BIC          : -
  R-squared    : 0.06851       R-sq (adj)   : 0.06723
  F-stat (adj) : 15.35094      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t         P>|t|   [0.025      0.975]   
  ------------  ---------  --------  --------  ------  ----------  ---------
  _intercept_   -36.66654  31.76115  -1.15445  0.2544  -100.63677  27.30370 
  BPXPLS        1.28087    0.40225   3.18430   0.0026  0.47071     2.09104  
  RIDAGEYR      0.61751    0.15848   3.89644   <0.001  0.29831     0.93670  
  RIAGENDR_2.0  -34.25273  7.46440   -4.58881  <0.001  -49.28681   -19.21866
  BMXBMI        2.13018    0.34941   6.09650   <0.001  1.42643     2.83393  

=== calibration: pulse ~ free T3 (descriptive, not part of the formal multiplicity family) ===
GLM: Gaussian

C:\Users\CONNOR\AppData\Local\Temp\ipykernel_11600\3574397299.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['wt_pooled'] = df['wt_thyroid_2yr'] / weight_divisor


In [4]:
# H2 population -> pulse and cholesterol vs metabolic syndrome
h2 = enrich('../data/processed/h2_analytic_sample.csv', 3)
h2_nobb = h2[~h2['on_beta_blocker']]

design2 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wt_pooled')

sample2_nobb = svy.Sample(pl.from_pandas(h2_nobb), design=design2)
model_pulse_mets = svy.GLM(sample2_nobb)
model_pulse_mets.fit(y='metabolic_syndrome', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='binomial')
print('=== pulse -> metabolic syndrome ===')
print(model_pulse_mets)

# Cholesterol models exclude anyone on a lipid-lowering drug, whose cholesterol
# reflects the medication rather than thyroid status.
h2_nolipid = h2[~h2['on_antihyperlipidemic']]
sample2_nolipid = svy.Sample(pl.from_pandas(h2_nolipid), design=design2)
model_chol_mets = svy.GLM(sample2_nolipid)
model_chol_mets.fit(y='metabolic_syndrome', x=['LBXTC', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='binomial')
print()
print('=== cholesterol -> metabolic syndrome ===')
print(model_chol_mets)

C:\Users\CONNOR\AppData\Local\Temp\ipykernel_11600\3574397299.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['wt_pooled'] = df['wt_thyroid_2yr'] / weight_divisor


=== pulse -> metabolic syndrome ===
GLM: Binomial (logit)
  Modeling : metabolic_syndrome
  n            : 2909          DF Residuals : 45
  Deviance     : 2748.0266     Scale        : 1.0000
  AIC          : 2769.3643     BIC          : -
  R-squared    : 0.23693       R-sq (adj)   : 0.23588
  F-stat (adj) : 38.59458      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.      Std.Err.  t          P>|t|   [0.025     0.975]  
  ------------  ---------  --------  ---------  ------  ---------  --------
  _intercept_   -10.15555  0.90581   -11.21151  <0.001  -11.97995  -8.33114
  BPXPLS        0.02566    0.00646   3.97257    <0.001  0.01265    0.03867 
  RIDAGEYR      0.04807    0.00455   10.56495   <0.001  0.03890    0.05723 
  RIAGENDR_2.0  -0.32801   0.17776   -1.84529   0.0716  -0.68603   0.03001 
  BMXBMI        0.18654    0.01686   11.06334   <0.001  0.15258    0.22050 



=== cholesterol -> metabolic syndrome ===
GLM: Binomial (logit)
  Modeling : metabolic_syndrome
  n            : 2743          DF Residuals : 45
  Deviance     : 2543.4979     Scale        : 1.0000
  AIC          : 2561.9193     BIC          : -
  R-squared    : 0.23135       R-sq (adj)   : 0.23023
  F-stat (adj) : 49.40726      Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.     Std.Err.  t          P>|t|   [0.025     0.975]  
  ------------  --------  --------  ---------  ------  ---------  --------
  _intercept_   -9.54218  0.60776   -15.70065  <0.001  -10.76626  -8.31809
  LBXTC         0.00661   0.00190   3.47396    0.0011  0.00278    0.01045 
  RIDAGEYR      0.04445   0.00528   8.42232    <0.001  0.03382    0.05509 
  RIAGENDR_2.0  -0.31665  0.15539   -2.03782   0.0475  -0.62962   -0.00368
  BMXBMI        0.18755   0.01555   12.05842   <0.001  0.15622    0.21888 


In [5]:
# H3 population -> pulse and cholesterol vs log-CRP
h3 = enrich('../data/processed/h3_analytic_sample.csv', 2)
h3['log_crp'] = np.log(h3['LBXCRP'])
h3_nobb = h3[~h3['on_beta_blocker']]

design3 = svy.Design(stratum='SDMVSTRA', psu='SDMVPSU', wgt='wt_pooled')

sample3_nobb = svy.Sample(pl.from_pandas(h3_nobb), design=design3)
model_pulse_crp = svy.GLM(sample3_nobb)
model_pulse_crp.fit(y='log_crp', x=['BPXPLS', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print('=== pulse -> log-CRP ===')
print(model_pulse_crp)

h3_nolipid = h3[~h3['on_antihyperlipidemic']]
sample3_nolipid = svy.Sample(pl.from_pandas(h3_nolipid), design=design3)
model_chol_crp = svy.GLM(sample3_nolipid)
model_chol_crp.fit(y='log_crp', x=['LBXTC', 'RIDAGEYR', svy.Cat('RIAGENDR', ref=1), 'BMXBMI'], family='gaussian')
print()
print('=== cholesterol -> log-CRP ===')
print(model_chol_crp)

=== pulse -> log-CRP ===
GLM: Gaussian (identity)
  Modeling : log_crp
  n            : 5310          DF Residuals : 28
  Deviance     : 6442.5289     Scale        : 211.3007
  AIC          : 16113.6206    BIC          : -
  R-squared    : 0.28241       R-sq (adj)   : 0.28187
  F-stat (adj) : 285.54731     Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.     Std.Err.  t          P>|t|   [0.025    0.975]  
  ------------  --------  --------  ---------  ------  --------  --------
  _intercept_   -6.28019  0.14642   -42.89216  <0.001  -6.58011  -5.98027
  BPXPLS        0.01761   0.00150   11.76948   <0.001  0.01455   0.02068 
  RIDAGEYR      0.00842   0.00157   5.37752    <0.001  0.00521   0.01163 
  RIAGENDR_2.0  0.25706   0.04402   5.84004    <0.001  0.16689   0.34722 
  BMXBMI        0.09236   0.00355   25.99140   <0.001  0.08508   0.09964 



=== cholesterol -> log-CRP ===
GLM: Gaussian (identity)
  Modeling : log_crp
  n            : 5123          DF Residuals : 28
  Deviance     : 6507.2021     Scale        : 205.7714
  AIC          : 15782.5885    BIC          : -
  R-squared    : 0.26914       R-sq (adj)   : 0.26857
  F-stat (adj) : 221.15631     Prob (F-adj) : <0.001

Coefficients:
  Term          Coef.     Std.Err.  t          P>|t|   [0.025    0.975]  
  ------------  --------  --------  ---------  ------  --------  --------
  _intercept_   -5.49940  0.13661   -40.25582  <0.001  -5.77923  -5.21956
  LBXTC         0.00179   0.00047   3.79444    <0.001  0.00082   0.00275 
  RIDAGEYR      0.00932   0.00180   5.17125    <0.001  0.00563   0.01301 
  RIAGENDR_2.0  0.28589   0.05060   5.64999    <0.001  0.18224   0.38954 
  BMXBMI        0.09744   0.00353   27.57249   <0.001  0.09020   0.10467 


C:\Users\CONNOR\AppData\Local\Temp\ipykernel_11600\3574397299.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['wt_pooled'] = df['wt_thyroid_2yr'] / weight_divisor
C:\Users\CONNOR\AppData\Local\Temp\ipykernel_11600\1707418410.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  h3['log_crp'] = np.log(h3['LBXCRP'])


In [6]:
# Multiple comparison correction for the H5 exploratory family, which is kept separate
# from H1-H4's correction. Mixing a pre-registered confirmatory set with
# a post-hoc exploratory one would muddle what each corrected p-value means.
tests = ['pulse -> triglycerides', 'pulse -> MetS', 'pulse -> log-CRP', 'cholesterol -> MetS', 'cholesterol -> log-CRP']
def p_value(model, term):
    return model.to_polars().filter(pl.col('term') == term)['p_value'][0]

fitted = [(model_pulse_tg, 'BPXPLS'), (model_pulse_mets, 'BPXPLS'), (model_pulse_crp, 'BPXPLS'),
          (model_chol_mets, 'LBXTC'), (model_chol_crp, 'LBXTC')]
p_values = [p_value(m, term) for m, term in fitted]

reject_fdr, p_values_fdr, _, _ = multipletests(p_values, alpha=0.05, method='fdr_bh')
reject_bonf, p_values_bonf, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

for t, p, rf, pf, rb, pb in zip(tests, p_values, reject_fdr, p_values_fdr, reject_bonf, p_values_bonf):
    print(f'{t}: raw p={p:.3g} | FDR-adj p={pf:.3g} (sig={rf}) | Bonferroni-adj p={pb:.3g} (sig={rb})')

pulse -> triglycerides: raw p=0.00263 | FDR-adj p=0.00263 (sig=True) | Bonferroni-adj p=0.0132 (sig=True)
pulse -> MetS: raw p=0.000254 | FDR-adj p=0.000635 (sig=True) | Bonferroni-adj p=0.00127 (sig=True)
pulse -> log-CRP: raw p=2.35e-12 | FDR-adj p=1.18e-11 (sig=True) | Bonferroni-adj p=1.18e-11 (sig=True)
cholesterol -> MetS: raw p=0.00115 | FDR-adj p=0.00143 (sig=True) | Bonferroni-adj p=0.00573 (sig=True)
cholesterol -> log-CRP: raw p=0.000727 | FDR-adj p=0.00121 (sig=True) | Bonferroni-adj p=0.00364 (sig=True)
